<div class="alert alert-block alert-info">
<b>1. Train Test Splitting <b>
</div>

In [66]:
import pandas as pd

df = pd.read_csv("car_sales_data.csv")
df

,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage,Price
0,Ford,Fiesta,1.0,Petrol,2002,127300,3074
1,Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
2,Ford,Mondeo,1.6,Diesel,2014,39190,24072
3,Toyota,RAV4,1.8,Hybrid,1988,210814,1705
4,VW,Polo,1.0,Petrol,2006,127869,4101
...,...,...,...,...,...,...,...
49995,BMW,M5,5.0,Petrol,2018,28664,113006
49996,Toyota,Prius,1.8,Hybrid,2003,105120,9430
49997,Ford,Mondeo,1.6,Diesel,2022,4030,49852
49998,Ford,Focus,1.0,Diesel,2016,26468,23630


In [67]:
df.columns

Index(['Manufacturer', 'Model', 'Engine size', 'Fuel type',
       'Year of manufacture', 'Mileage', 'Price'],
      dtype='object')

In [68]:
# X = All columns except 'target' which is price
# Y = 'target' column which is price

X = df.drop('Price', axis=1)            # dependent columns which will be used to predict the target column
Y = df['Price']                         # target column to be predicted using the dependent columns

In [69]:
X.head(1)

,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage
0,Ford,Fiesta,1.0,Petrol,2002,127300


In [70]:
Y.head(1)

0    3074
Name: Price, dtype: int64

In [71]:
from sklearn.model_selection import train_test_split
train_X, test_X, train_Y, test_Y = train_test_split(X, Y, test_size=0.2, random_state=42)

# train_X, test_X, train_Y, test_Y = train_test_split(X, Y, test_size=0.2, shuffle=False)

In [72]:
train_X

,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage
39087,Ford,Fiesta,1.4,Petrol,1990,143415
30893,VW,Polo,1.0,Petrol,1990,259900
45278,Toyota,RAV4,2.0,Hybrid,2006,106750
16398,Toyota,RAV4,1.8,Hybrid,2001,126649
13653,Porsche,718 Cayman,2.0,Petrol,1992,66179
...,...,...,...,...,...,...
11284,Toyota,Yaris,1.2,Hybrid,1984,279567
44732,VW,Golf,1.2,Diesel,2010,108027
38158,VW,Passat,2.0,Diesel,2016,72963
860,BMW,X3,2.4,Diesel,1988,275595


In [73]:
print(train_X.shape, train_Y.shape)
print(test_X.shape, test_Y.shape)

(40000, 6) (40000,)
(10000, 6) (10000,)


<div class="alert alert-block alert-info">
<b>2. Model Training/Applying Classifier  <b>
</div>

In [74]:
categorical_cols = train_X.select_dtypes(include=['object']).columns
print(categorical_cols)

Index(['Manufacturer', 'Model', 'Fuel type'], dtype='object')


In [75]:
train_X.info()
train_Y.value_counts().head()
train_X.head()

<class 'pandas.core.frame.DataFrame'>
Index: 40000 entries, 39087 to 15795
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Manufacturer         40000 non-null  object 
 1   Model                40000 non-null  object 
 2   Engine size          40000 non-null  float64
 3   Fuel type            40000 non-null  object 
 4   Year of manufacture  40000 non-null  int64  
 5   Mileage              40000 non-null  int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 2.1+ MB


,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage
39087,Ford,Fiesta,1.4,Petrol,1990,143415
30893,VW,Polo,1.0,Petrol,1990,259900
45278,Toyota,RAV4,2.0,Hybrid,2006,106750
16398,Toyota,RAV4,1.8,Hybrid,2001,126649
13653,Porsche,718 Cayman,2.0,Petrol,1992,66179


In [76]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Columns
cat_cols = ['Manufacturer', 'Model', 'Fuel type']
num_cols = ['Engine size', 'Year of manufacture', 'Mileage']

# Preprocessor using OrdinalEncoder (FAST)
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Pipeline
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('rf', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model.fit(train_X, train_Y)

# Predict
model_pred = model.predict(test_X)
model_pred


array([67860.07 , 35459.515, 17166.345, ..., 39748.59 ,   581.685,
       32985.22 ], shape=(10000,))

In [77]:
test_X.dtypes

Manufacturer            object
Model                   object
Engine size            float64
Fuel type               object
Year of manufacture      int64
Mileage                  int64
dtype: object

In [78]:
test_X.head()

,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage
33553,Toyota,RAV4,2.4,Hybrid,2020,21317
9427,Ford,Focus,1.8,Petrol,2018,22500
199,Ford,Mondeo,1.6,Diesel,2013,79521
12447,Toyota,Yaris,1.2,Petrol,2003,159534
39489,Ford,Mondeo,1.4,Diesel,2000,126511


In [79]:
model_pred = model.predict(test_X)
model_pred

array([67860.07 , 35459.515, 17166.345, ..., 39748.59 ,   581.685,
       32985.22 ], shape=(10000,))

In [80]:
#R2 score
# Accuracy Score
from sklearn.metrics import r2_score

pred_Y = model.predict(test_X)
r2 = r2_score(test_Y, pred_Y)
print("R2 Score:", r2)

R2 Score: 0.9982705980187699


In [81]:
from sklearn.metrics import accuracy_score
import numpy as np
model_pred = model.predict(test_X)
# print("Accuracy Score:", model_acc_svc)
model_pred_rounded = np.round(model_pred * 100, 3)
print(model_pred_rounded)

[6786007.  3545951.5 1716634.5 ... 3974859.    58168.5 3298522. ]


In [82]:
import pickle
pickle.dump(model, open('model.pkl', 'wb'))

In [83]:
model_svc = pickle.load(open('model.pkl', 'rb'))